<a href="https://colab.research.google.com/github/kumarsirish/FDP-AGENENTIC-AI-RAG/blob/main/rag-agentic-01/fictional-depatment-rag-agentic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Agentic RAG

This notebook walks through building an Agentic RAG system using LangChain and FAISS.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## 1. Install Dependencies

In [ ]:
import os

if not os.path.exists('requirements.txt'):
    ! wget https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-agentic-01/requirements.txt
else:
    print('requirements.txt already exists.')
%pip install -r  requirements.txt


## 2. Set API Keys

In [ ]:
import os
from dotenv import load_dotenv
#from kaggle_secrets import UserSecretsClient
from google.colab import userdata

# Fallback: try Google Colab secrets
#try:
#    from google.colab import userdata
#    if not os.getenv("HF_TOKEN"):
HF_TOKEN = userdata.get("HF_TOKEN").strip()
#    if not os.getenv("GEMINI_API_KEY"):
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY").strip()
#except ImportError:
#pass


# Load from .env file (local development)
# load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")


#Kaggle
#user_secrets = UserSecretsClient()
#HF_TOKEN = user_secrets.get_secret("HF_TOKEN").strip()
#GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

#HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
#GEMINI_API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip()


if not GEMINI_API_KEY or not HF_TOKEN:
    print("Warning: HF_TOKEN or GEMINI_API_KEY not set. Please set it in .env file or Colab secrets.")

In [ ]:
department_info = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",

    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science.",

]


## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

documents = [Document(page_content=text) for text in department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

# Force anonymous download for public embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)

db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

In [ ]:
# Create retriever — returns top 2 most similar documents for any query
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})
print("Retriever ready.")

## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [ ]:
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_react_agent, AgentExecutor # Updated import
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # Added MessagesPlaceholder


# ── 1. Load vector store ──────────────────────────────────────────────

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    #If allow_dangerous_deserialization is set to False,
    #FAISS.load_local will refuse to load any serialized data
    #that it deems potentially unsafe.
    allow_dangerous_deserialization=True
)
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# ── 2. Tool ───────────────────────────────────────────────────────────
@tool
def search_answers(query: str) -> str:
    """Search the knowledge base for information about the DQE department."""
    results = retriever.invoke(query)
    docs = "\n".join([doc.page_content for doc in results])
    print(f"\n[TOOL CALLED] query='{query}'")
    print(f"[RETRIEVED]:\n{docs}\n")
    return docs if docs.strip() else "No relevant documents found."



llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    provider="auto",           # ← KEY FIX: routes to Nebius/Fireworks/etc.
    huggingfacehub_api_token=os.getenv("HF_TOKEN"), # Use os.getenv for HF_TOKEN
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1,
)


# Wrap in ChatHuggingFace — this enables bind_tools() for agentic RAG
chat_llm = ChatHuggingFace(llm=llm_endpoint, verbose=True)


# ── 4. Agent ──────────────────────────────────────────────────────────
# Define the prompt for the agent
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant for the Department of Quantum Engineering (DQE). "
     "You answer questions using the knowledge base via the search_answers tool. "
     "\n\nHere are the tools you can use:\n{tools}\n\nRules:"
     "\n- Always use search_answers when the question is about DQE — its people, courses, research, or policies."
     "\n- Do NOT use the tool for general knowledge questions (e.g., 'what is quantum computing?')."
     "\n- If the tool returns no results, say you don't have that information in the knowledge base."
     "\n- Answer concisely based only on what the tool returns. Do not hallucinate details.\n"
     "Use the following format:\n"
     "Question: the input question you must answer\n"
     "Thought: you should always think about what to do\n"
     "Action: the action to take, should be one of [{tool_names}]\n"
     "Action Input: the input to the action\n"
     "Observation: the result of the action\n"
     "... (this Thought/Action/Action Input/Observation can repeat N times)\n"
     "Thought: I now know the final answer\n"
     "Final Answer: the final answer to the original input question"),
    MessagesPlaceholder("chat_history"), # Use MessagesPlaceholder conversational memory across multiple turns
    ("human", "{input}\n{agent_scratchpad}") # Combine human input and scratchpad (which contains all previous thoughts, actions, and observations as a string
])


# Create the ReAct agent
agent = create_react_agent(chat_llm, [search_answers], prompt)

# Create the AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=[search_answers], verbose=True)

print(f"Agent ready ✅")

## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [ ]:
def ask(question: str) -> str:
    response = agent_executor.invoke({"input": question, "chat_history": []})
    # The final answer from AgentExecutor is in the 'output' key
    return response["output"]

print("Answer: ",ask("What is DQE?"))

In [ ]:
print("Answer: ",ask("How many students does DQE have?"))

In [ ]:
print("Answer: ",ask("Whats the objective?"))